In [0]:
# IMPORTANDO A BIBLIOTECA NECESSÁRIA PARA MANIPULAÇÃO DOS DADOS, MUDANDO O TEXTO PARA MINUSCULO
from pyspark.sql.functions import col, lower, initcap, month, year, when, count, regexp_replace
from pyspark.sql.types import DoubleType, IntegerType

In [0]:
# CRIANDO O DATAFRAME COM OS DADOS
df_bronze = spark.table(
    "projeto_profissional_bigdata.datatran_2018.bronze_datatran18"
)

In [0]:
# MOSTRANDO OS TIPOS DE DADOS COM SCHEMA
df_bronze.printSchema()

In [0]:
# MOSTRANDO OS DADOS UTILIZANDO O SELECT PARA VIZUALIZAÇÃO DOS RESULTADOS
df_silver = df_bronze.select(
    "id",
    "data_inversa",
    "uf",
    "br",
    "km",
    "municipio",
    "causa_acidente",
    "tipo_acidente"
)

df_silver.show(5, truncate=False)

In [0]:
#RETIRANDO OS VALORES NULOS DA TABELA
df_silver = df_silver.dropna(
    subset = [
        "id",
        "data_inversa",
        "uf",
        "br",
        "km",
        "municipio",
        "causa_acidente",
        "tipo_acidente"
    ]
)

df_silver.show(5, truncate=False)


In [0]:
# RENOMEANDO A COLUNA ( 1º ORIGINAL, 2 NOVA)
df_silver = df_silver.withColumnRenamed("data_inversa","data_acidente")

df_silver.show(5, truncate = False)

In [0]:
# PADRONIZANDO OS DADOS PARA MINUSCULO
df_silver = df_silver.withColumn(
    "municipio",
    initcap(col("municipio"))
)

df_silver = df_silver.withColumn(
    "causa_acidente",
    lower(col("causa_acidente"))
)

df_silver = df_silver.withColumn(
    "tipo_acidente",
    lower(col("tipo_acidente"))
)

df_silver.show(5, truncate = False)

In [0]:
# CRIANDO COLUNA MÊS ACIDENTE
df_silver = df_silver.withColumn(
    "mes_acidente",
    month(col("data_acidente"))
)

In [0]:
# CRIANDO A COLUNA ANO ACIDENTE
df_silver = df_silver.withColumn(
    "ano_acidente",
    year(col("data_acidente"))
)

df_silver.show(5, truncate = False)

In [0]:
# RIANDO A COLUNA NOME_MES USANDO A COLUNA MES_ACIDENTE COMO REFERENCIA

df_silver = df_silver.withColumn(
    "nome_mes",
    when(col("mes_acidente") == 1, "Janeiro")
    .when(col("mes_acidente") == 2, "Fevereiro")
    .when(col("mes_acidente") == 3, "Março")
    .when(col("mes_acidente") == 4, "Abril")
    .when(col("mes_acidente") == 5, "Maio")
    .when(col("mes_acidente") == 6, "Junho")
    .when(col("mes_acidente") == 7, "Julho")
    .when(col("mes_acidente") == 8, "Agosto") 
    .when(col("mes_acidente") == 9, "Setembro")
    .when(col("mes_acidente") == 10, "Outubro")
    .when(col("mes_acidente") == 11, "Novembro")
    .when(col("mes_acidente") == 12, "Dezembro")
)

df_silver.show(5, truncate = False)

In [0]:
#CONTANDO REGISTROS DA COLUNA BR = "NA"

df_silver.filter(col("br") == "NA").count()

In [0]:
#TROCANDO O VALOR DA COLUNA "BR" DE "NA" = None

df_silver = df_silver.withColumn(
    "br",
    when(col("br") == "NA", None)
    .otherwise(col("br"))
)

In [0]:
#LIMPANDO OS REGISTROS NULOS DA COLUNA "BR"

df_silver = df_silver.dropna(subset=["br"])

In [0]:
#CONVERTENDO A COLUNA BR PARA PARA O TIPO INTEGER
df_silver = df_silver.withColumn(
    "br",
    col("br").cast((IntegerType()))
)

df_silver.printSchema()
df_silver.show(5)

In [0]:
#TRANSFORMANDO A COLUNA "KM" PARA O TIPO DOUBLE E TROCANDO OS CARACTERES "," E POR "."

df_silver = df_silver.withColumn(
    "km",
    regexp_replace(col("km"), ",",".").cast(DoubleType())
)

df_silver.printSchema()
df_silver.show(5)


In [0]:
#SALVANDO A TEBELA DELTA = SILVER
df_silver.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("projeto_profissional_bigdata.datatran_2018.silver_datatran18")

In [0]:
#BUSCANDO A TABELA SILVER UTILIZANDO PYTHON

df = spark.table("projeto_profissional_bigdata.datatran_2018.silver_datatran18")

df.show()

In [0]:
#BUSCANDO A TABELA SILVER UTILIZANDO SQL

%sql
SELECT * FROM projeto_profissional_bigdata.datatran_2018.silver_datatran18